In [ ]:
import os, sys
sys.path.append(os.path.abspath(".."))
from mienc.support import single_iter, correct_vector, get_pool
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from mienc.corrector import Corrector
from mienc.support import pair_mutual_information, task_producer, total_mutual_information
import multiprocessing as mp

In [ ]:
nsamples = 0
steps = 200
iterations = 10000
workers = 23
bins=8
samples=400

In [ ]:
increment = 1 / steps

true_value = -0.5 * \
    np.log(1 - (np.arange(steps) / steps) ** 2)
correction = np.zeros(steps)

with get_pool(workers) as pool:
    for i in tqdm(range(0,200,10), desc="Step"):
        means = 0, 0
        correlation_matrix = [[1, i * increment], [i * increment, 1]]
        I = pool.map(
            single_iter,
            (
                (means, correlation_matrix, samples, bins)
                for __ in range(iterations)
            ),
        )
        plt.hist(I, "auto")
        plt.ylim(plt.ylim())
        plt.vlines([np.mean(I), np.median(I)], *plt.ylim(),"k", ["solid", "dashdot"])
        plt.title(round(i*increment,3))
        plt.xlabel("MI")
        plt.show()

In [ ]:
r=0.85
S = 50
cov=np.full((S,S),r)
np.fill_diagonal(cov,1)
def one_iter (x):
    return x, [total_mutual_information(V, 8) for V in task_producer(np.random.multivariate_normal(np.zeros(S),cov, 400),99,True)]
results = np.empty((100,int(S*(S-1)/2),1000))
with mp.Pool(23) as Polla:
    for i, R in tqdm(Polla.imap(one_iter, range(1000)), total=1000):
        results[:,:,i] = R

In [ ]:
corrector = Corrector(8,400,200,50000,400,cache_dir="../cache/", config="../config.ini", verbose=True)
corrector.compute_correction()
corrected = corrector.correct(results.reshape((100,-1)))

In [ ]:
print(f"True: {np.mean(corrected[0,:]):.3}+/-{np.std(corrected[0,:]):.3}")
print(f"Surr: {np.mean(corrected[1:,:]):.3}+/-{np.std(corrected[1:,:]):.3}")

In [ ]:
import nilearn

# Is the problem in surrogation?

In [ ]:
import io, h5py, zipfile
import numpy as np
from tqdm import tqdm
import pickle
from scipy import signal
import os, sys
sys.path.append(os.path.abspath(".."))
from mienc.support import surrogate
from mienc import NonLinearEstimator
from collections import Counter
import matplotlib.pyplot as plt
from scipy.io import loadmat

bands={i:b for i, b in enumerate([[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]], start=1)}

basePath = "../NonLinearityData/EEG_el_so_BLP_NEW"
f_sample = 1000
fs_under = 120

def surrogate_piece (piece):
    piece_surrogate = surrogate(piece)
    chunks = int(piece.shape[0]/fs_under)
    tot_samples_dest = chunks * fs_dest
    border = max(1, tot_samples_dest//20)
    return signal.resample(piece_surrogate[:fs_under*chunks,:], tot_samples_dest)[border:-border,:]


def fs_band(band):
    return int(bands[band][1]*2*1.125+0.5)

for band in range(9):
    with open(f"../auxiliary_data/new_pieces_band{band+1}.pickle", "rb") as fp:
        pass

In [ ]:
zip_file = zipfile.PyZipFile("../../NonLinearityData/EEG_el_so_BLP_20230714.zip")
subjects_shapes = []
band = 0
for subject in tqdm(range(50), desc="Subject"):
    archive = h5py.File(io.BytesIO(zip_file.read(f"Sub_{subject+1}/EEG_bands.mat")))
    band_data = archive[archive["EEG_bands"][0,band]]
    tmp = []
    for i in range(band_data.shape[0]):
        tmp.append(archive[band_data[i,0]].shape[0])
    subjects_shapes.append(tmp[:])


In [ ]:
max_per_sub = [np.max(a) for a in subjects_shapes]
ranked = sorted(max_per_sub)[::-1]
amount = [a*i for i,a in enumerate(ranked, start=1)]
plt.plot(np.arange(len(amount)), amount)
plt.xlabel("# of subjects")
plt.ylabel("Size of the data")
plt.show()

In [ ]:
np.array(amount)[np.argsort(amount)[-6:]],np.argsort(amount)[-6:], ranked[-8:], 9*60, 500/9*1000

In [ ]:
import os, sys
sys.path.append(os.path.abspath(".."))
from mienc import NonLinearEstimator as NLE
from scipy.io import loadmat
import numpy as np

for band in range(1, 10):
    samples = loadmat(f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/trimmed_EEG_fixedTime_band{band}.mat")["EEG"].shape[0]
    bins = max(8, int(np.power(samples, 1/3)))
    nle = NLE(config_file="../config.ini", bins=bins, surrogates=99, verbose=True,
              cache="../cache/", save_out=True, dataset="trimmed_EEG_bands_time", suffix="Extended", workers=23)
    nle.estimate(display=False, dataset_sub=str(band), extended_stats=True, compute_shadow="extend")
    del nle

In [ ]:
bands={i:b for i, b in enumerate([[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]], start=1)}
def fs_band(band):
    return int(bands[band][1]*2*1.125+0.5)


results = {}

for band in range(1,8):
    samples=45*fs_band(band)
    bins = max(8, int(np.power(samples, 1/3)))
    nle = NLE(config_file="../config.ini", bins=bins, surrogates=99, verbose=True,
                cache="../cache/", save_out=False, dataset="trimmed_EEG_bands_time", suffix="Cutting", workers=23)
    results[band] = nle.estimate(display=False, dataset_sub="7", extended_stats=True, compute_shadow=True, truncate_input=samples)
    del nle

In [ ]:
44*2*1.125*45

In [ ]:
import os, sys
sys.path.append(os.path.abspath(".."))
from mienc import NonLinearEstimator as NLE

nle = NLE(config_file="../config.ini", bins=8, surrogates=99, cache="../cache", save_out=True, dataset="eso245_aal",workers=23, suffix="ExtSha", verbose=True)

gotcha2 = nle.estimate(display=False, dataset_sub="strin", extended_stats=True, compute_shadow=True)

In [ ]:
gotcha.keys()

In [ ]:
rnl = [np.average(1-results[band]["gaussMI"]/results[band]["totalMI"]) for band in results]
rnl_sha = [np.average(1-results[band]["gaussMIshadow"]/results[band]["totalMIshadow"]) for band in results]
sr = [4,8,12,15,18,30,44]
plt.plot(sr,rnl, label="True")
plt.plot(sr,rnl_sha, label="Shadow")
plt.legend(loc=0)
plt.xscale("log")
plt.yscale("log")
plt.show()
np.polyfit(np.log10(sr), np.log10(rnl_sha),1)

In [ ]:
import numpy as np


def surrogate(x:np.ndarray, multivariate:bool=True, extension:int=1)->np.ndarray:
    """Generate a common angle surrogate of a N-D array (surrogates the first axis).
        Input:
        x: an N-dimensional np.Array with time along the first axis (time x whatever x ... x whatever).
        multivariate: if True (default) applies the same random phases to all the series.
        extension: create a longer surrogate by joining extension many.
        Output:
        np.array containing the surrogate time series such that output shape matches input shape."""
    if x.shape[1] > x.shape[0]:
        warnings.warn(
            "It looks you have more series than timepoints, or maybe you should transpose the input.", RuntimeWarning)
    fft = np.fft.rfft(x, axis=0)
    fftX1 = []
    extra_shape = [1] if multivariate else x.shape[1:]

    for i in range(extension):
        rpha = np.exp(
            2 * np.pi * np.random.rand(int(x.shape[0] / 2 + 1), *extra_shape) * 1.0j)
        fftX1.append(fft*rpha)
            
    xs = np.concatenate([np.fft.irfft(tmp, n=x.shape[0]*extension, axis=0) for tmp in fftX1],0)
    return xs

In [ ]:
a = np.array([[1,1,1,1,1,1,1],[2,2,2,2,2,2,2],[3,3,3,3,3,3,3]]).T
surrogate_extend(a,False,2)

In [ ]:
import os, sys
sys.path.append(os.path.abspath(".."))
from mienc.support import surrogate, surrogate_extend
import numpy as np

a = np.array([[1,0,1,0,1,0,1,0],[2,1,0,1,2,1,0,1],[3,2,1,2,3,2,1,2]]).T
ext=2
np.random.seed(42)
print(surrogate(a,True))
np.random.seed(42)
print(surrogate_extend(a,True,ext).shape)
np.random.seed(42)
print(surrogate_extend(a,True,ext))


In [ ]:
# import pickle
# with open("../auxiliary_data/trimmedEEG_band7_trucation.pkl", "wb") as fp:
#     pickle.dump(results, fp)

In [ ]:
zip_file = zipfile.PyZipFile("../NonLinearityData/EEG_el_so_BLP_20230714.zip")
new_pieces = [[[] for subject in range(50)] for band in range(9)]
for subject in tqdm(range(50), desc="Subject"):
    archive = h5py.File(io.BytesIO(zip_file.read(f"Sub_{subject+1}/EEG_bands.mat")))

    for band in tqdm(bands, desc="Band", total=9):
        band_data = archive[archive["EEG_bands"][0,band-1]]
        fs_dest = fs_band(band)
        needed_samples = max(1000, 45*fs_dest)
        pieces = []
        tot_len = 0

        for i in range(band_data.shape[0]):
            this_piece = archive[band_data[i,0]]
            chunks = int(this_piece.shape[0]/f_sample)

            if chunks == 0:
                continue

            tot_samples_dest = chunks * fs_dest
            border = max(1, tot_samples_dest//20)
            tot_len += tot_samples_dest - 2*border
            
            tot_samples_under = chunks * fs_under
            blaf = signal.resample(this_piece[:f_sample*chunks,:], tot_samples_under)
            new_pieces[band-1][subject].append(blaf)
            if tot_len > needed_samples:
                break

for band in range(9):
    with open(f"auxiliary_data/new_pieces_band{band+1}.pickle", "wb") as fp:
        pickle.dump(new_pieces[band], fp)

In [ ]:
for band in tqdm(bands, desc="Band", total=9):
    if band==1:
        continue
    with open(f"../auxiliary_data/new_pieces_band{band}.pickle", "rb") as fp:
        new_pieces = pickle.load(fp)
    for surros in tqdm(range(5)):    
        fs_dest = fs_band(band)
        tmp = [np.concatenate([surrogate_piece(piece) for piece in new_pieces[subject]])[:max(1000, 45*fs_dest),:] for subject in range(50)]
        in_array = np.transpose(np.array(tmp),(1,2,0))

        bins = max(8, int(np.power(45*fs_dest, 1/3)))

        nle = NonLinearEstimator("../config.ini", bins=bins, surrogates=99*(not surros), cache="../cache/", save_out="EEG_voltage_piecewiseSurrogates_fixedTime", suffix=f"S{surros:02}B{band}")
        nle.estimate(in_array[:45*fs_dest,:,:], extended_stats=True, compute_shadow=False,)

        nle = NonLinearEstimator("../config.ini", bins=10, surrogates=99*(not surros), cache="../cache/", save_out="EEG_voltage_piecewiseSurrogates_fixedSamples", suffix=f"S{surros:02}B{band}")
        nle.estimate(in_array[:1000,:,:], extended_stats=True, compute_shadow=False,)


In [ ]:
with open(f"../auxiliary_data/new_pieces_band{band}.pickle", "rb") as fp:
    new_pieces = pickle.load(fp)
numpieces = [len(p) for p in new_pieces]
plt.hist(numpieces)
plt.show()
Counter(numpieces).most_common()

In [ ]:
np.median(numpieces), np.mean(numpieces)

In [ ]:
for band in tqdm(bands, desc="Band", total=9):
    mat = loadmat("/home/raffaelli/NonLinearity/NonLinearityData/iEEG/FIX_iEEG_fixedTime_band{}.mat".format(band))["EEG"]
    time, electrodes, subjects = mat.shape

    new_subj = np.empty_like(mat)
    new_surr = np.empty_like(mat)

    for subj in range(subjects):
        split = np.random.randint(100, time-100)
        new_subj[:time-split,:,subj] = mat[split:,:,subj]
        new_surr[:time-split,:,subj] = surrogate(mat[split:,:,subj])
        new_subj[time-split:,:,subj] = mat[:split,:,subj]
        new_surr[time-split:,:,subj] = surrogate(mat[:split,:,subj])

    fs_dest = fs_band(band)
    bins = max(8, int(np.power(45*fs_dest, 1/3)))


    nle = NonLinearEstimator("../config.ini", bins=bins, surrogates=99, cache="../cache/", save_out="iEEG_voltage_piecewiseSurrogates_fixedTime", suffix=f"T00B{band}")
    nle.estimate(new_subj, extended_stats=True, compute_shadow=False,)

    nle = NonLinearEstimator("../config.ini", bins=bins, surrogates=99, cache="../cache/", save_out="iEEG_voltage_piecewiseSurrogates_fixedTime", suffix=f"S00B{band}")
    nle.estimate(new_surr, extended_stats=True, compute_shadow=False,)


In [ ]:
"/home/raffaelli/NonLinearity/NonLinearityData/iEEG/FIX_iEEG_fixedTime_band{}.mat".format(band)

In [ ]:
import cProfile

def surroga ():
    tmp = [np.concatenate([surrogate_piece(piece) for piece in new_pieces[subject]])[:max(1000, 45*fs_dest),:] for subject in tqdm(range(5))]
    in_array = np.transpose(np.array(tmp),(1,2,0))
    print(in_array.shape)

cProfile.run("surroga()")

In [ ]:
tot_size=0
for subj in range(50):
    for piece in new_pieces[subj]:
        tot_size+=piece.size

print(piece.dtype, tot_size*8/1e9)

In [ ]:
qwe = new_pieces[subj][0]
qwe.shape

In [ ]:
bands[7]

In [ ]:
import matplotlib.pyplot as plt
band=7
with open(f"../auxiliary_data/new_pieces_band7.pickle", "rb") as fp:
    high_pieces = pickle.load(fp)
    


In [ ]:
fftX1 = np.fft.rfft(high_pieces[0][0][:,:10], axis=0)
df = 1000/high_pieces[0][0].shape[0]
plt.plot(fftX1[:, :10])
plt.ylim(plt.ylim())
plt.vlines([bands[band][0]/df,bands[band][1]/df], *plt.ylim(),"k","dashdot")
plt.xlim((bands[band][0]/df/2,bands[band][1]/df*1.5))
plt.ylim((-100,100))
plt.hlines([0],100,600,"r","dotted",lw=2)
plt.show()

In [ ]:
500*df